# Assignment 3: Generative Adversarial Text-to-Image Synthesis

This notebook implements a conditional GAN for Oxford-102 flower text-to-image synthesis. It is structured as a Colab-ready tutorial with separate training and testing sections.

Key constraints handled here:

- Source encoder is trained without class labels, using image reconstruction/GAN gradients plus label-free contrastive learning.
- Target generator consumes source image representations and text encodings.
- Generator parameter count is asserted to be no more than half of the source encoder parameter count.
- No pretrained model or checkpoint is used. The converted HDF5 dataset contains precomputed embeddings, but this notebook ignores them and trains the text encoder from raw captions.
- The split randomly selects 20 train classes and 5 test classes from Oxford-102 classes.


## Setup

In [ ]:
!pip -q install gdown h5py scikit-learn

In [ ]:
import io
import math
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import gdown
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("data")
CHECKPOINT_DIR = Path("checkpoints")
DATA_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)
print("Device:", DEVICE)


## Data Download and Split

The base repository for this assignment provides a converted Flowers HDF5 file. The file includes image bytes, class ids, raw text captions, and pretrained text embeddings. This notebook only reads image bytes, class ids, and raw text captions.

In [ ]:
FLOWERS_H5_PATH = DATA_DIR / "flowers.hdf5"
FLOWERS_H5_GDRIVE_ID = "1EgnaTrlHGaqK5CCgHKLclZMT_AMSTyh8"

if not FLOWERS_H5_PATH.exists():
    gdown.download(id=FLOWERS_H5_GDRIVE_ID, output=str(FLOWERS_H5_PATH), quiet=False)

assert FLOWERS_H5_PATH.exists(), "flowers.hdf5 was not downloaded. Re-run this cell in a network-enabled Colab runtime."


In [ ]:
def decode_h5_scalar(value):
    value = value[()]
    if isinstance(value, bytes):
        return value.decode("utf-8")
    if isinstance(value, np.bytes_):
        return value.decode("utf-8")
    if isinstance(value, np.ndarray):
        if value.shape == ():
            return decode_h5_scalar(value)
        value = value.tolist()
    if isinstance(value, list):
        if len(value) == 1:
            return str(value[0])
        return " ".join(map(str, value))
    return str(value)


def scan_h5_metadata(h5_path):
    classes = set()
    texts_by_class = defaultdict(list)
    counts_by_class = Counter()
    with h5py.File(h5_path, "r") as h5:
        for split in h5.keys():
            for key in h5[split].keys():
                example = h5[split][key]
                class_name = decode_h5_scalar(example["class"])
                text = decode_h5_scalar(example["txt"])
                classes.add(class_name)
                texts_by_class[class_name].append(text)
                counts_by_class[class_name] += 1
    return sorted(classes), texts_by_class, counts_by_class


all_classes, texts_by_class, counts_by_class = scan_h5_metadata(FLOWERS_H5_PATH)
assert len(all_classes) >= 25, f"Expected at least 25 classes, found {len(all_classes)}"

rng = random.Random(SEED)
selected_classes = rng.sample(all_classes, 25)
TRAIN_CLASSES = sorted(selected_classes[:20])
TEST_CLASSES = sorted(selected_classes[20:])
CLASS_TO_IDX = {class_name: idx for idx, class_name in enumerate(sorted(selected_classes))}

print("Train classes:", TRAIN_CLASSES)
print("Test classes:", TEST_CLASSES)
print("Selected examples:", sum(counts_by_class[c] for c in selected_classes))


In [ ]:
TOKEN_RE = re.compile(r"[a-z]+")


class Vocabulary:
    def __init__(self, texts, min_freq=1, max_size=6000):
        counts = Counter()
        for text in texts:
            counts.update(self.tokenize(text))
        words = [word for word, count in counts.most_common(max_size - 2) if count >= min_freq]
        self.itos = ["<pad>", "<unk>"] + words
        self.stoi = {word: idx for idx, word in enumerate(self.itos)}

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenize(text):
        return TOKEN_RE.findall(str(text).lower())

    def encode(self, text, max_len=48):
        ids = [self.stoi.get(token, 1) for token in self.tokenize(text)[:max_len]]
        ids += [0] * (max_len - len(ids))
        return torch.tensor(ids, dtype=torch.long)


selected_texts = []
for class_name in selected_classes:
    selected_texts.extend(texts_by_class[class_name])

VOCAB = Vocabulary(selected_texts, min_freq=1, max_size=6000)
print("Vocabulary size:", len(VOCAB))


In [ ]:
class OxfordFlowersTextDataset(Dataset):
    def __init__(self, h5_path, selected_classes, class_to_idx, vocab, training=True, image_size=64):
        self.h5_path = str(h5_path)
        self.selected_classes = set(selected_classes)
        self.class_to_idx = class_to_idx
        self.vocab = vocab
        self.training = training
        self.samples = []
        self._h5 = None

        with h5py.File(self.h5_path, "r") as h5:
            for split in h5.keys():
                for key in h5[split].keys():
                    example = h5[split][key]
                    class_name = decode_h5_scalar(example["class"])
                    if class_name in self.selected_classes:
                        text = decode_h5_scalar(example["txt"])
                        self.samples.append({"split": split, "key": key, "class": class_name, "text": text})

        self.base_transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])
        self.source_aug = T.Compose([
            T.RandomResizedCrop(image_size, scale=(0.80, 1.0)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
            T.ToTensor(),
            T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

    def _handle(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        meta = self.samples[idx]
        example = self._handle()[meta["split"]][meta["key"]]
        image_bytes = bytes(np.array(example["img"]))
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        tokens = self.vocab.encode(meta["text"])
        class_id = self.class_to_idx[meta["class"]]
        if self.training:
            view1 = self.source_aug(image)
            view2 = self.source_aug(image)
        else:
            view1 = self.base_transform(image)
            view2 = self.base_transform(image)
        return {
            "image": self.base_transform(image),
            "view1": view1,
            "view2": view2,
            "tokens": tokens,
            "class_id": torch.tensor(class_id, dtype=torch.long),
            "class_name": meta["class"],
            "text": meta["text"],
        }


train_dataset = OxfordFlowersTextDataset(
    FLOWERS_H5_PATH, TRAIN_CLASSES, CLASS_TO_IDX, VOCAB, training=True
)
test_dataset = OxfordFlowersTextDataset(
    FLOWERS_H5_PATH, TEST_CLASSES, CLASS_TO_IDX, VOCAB, training=False
)
all_selected_dataset = OxfordFlowersTextDataset(
    FLOWERS_H5_PATH, sorted(selected_classes), CLASS_TO_IDX, VOCAB, training=False
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available(), drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
all_selected_loader = DataLoader(all_selected_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

print("Train examples:", len(train_dataset))
print("Test examples:", len(test_dataset))


## Models

The source encoder is intentionally larger than the target generator. The assertion below enforces the generator-size constraint before training starts.

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=96, hidden_dim=128, output_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim, output_dim),
            nn.LayerNorm(output_dim),
            nn.Tanh(),
        )

    def forward(self, tokens):
        embedded = self.embedding(tokens)
        _, hidden = self.gru(embedded)
        return self.proj(hidden[-1])


class SourceEncoder(nn.Module):
    def __init__(self, representation_dim=256, projection_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(96),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(96, 192, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(192),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(192, 384, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(384),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(384, 768, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(768),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.representation = nn.Sequential(
            nn.Flatten(),
            nn.Linear(768, representation_dim),
            nn.LayerNorm(representation_dim),
            nn.Tanh(),
        )
        self.projection = nn.Sequential(
            nn.Linear(representation_dim, representation_dim),
            nn.ReLU(inplace=True),
            nn.Linear(representation_dim, projection_dim),
        )

    def forward(self, image, project=False):
        rep = self.representation(self.features(image))
        if project:
            return F.normalize(self.projection(rep), dim=1)
        return rep


class TargetGenerator(nn.Module):
    def __init__(self, representation_dim=256, text_dim=128):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(representation_dim + text_dim, 256 * 4 * 4),
            nn.BatchNorm1d(256 * 4 * 4),
            nn.ReLU(inplace=True),
        )
        self.net = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )

    def forward(self, source_representation, text_encoding):
        x = torch.cat([source_representation, text_encoding], dim=1)
        x = self.fc(x).view(x.size(0), 256, 4, 4)
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self, text_dim=128):
        super().__init__()
        self.image_features = nn.Sequential(
            nn.utils.spectral_norm(nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.utils.spectral_norm(nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.utils.spectral_norm(nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.utils.spectral_norm(nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1)),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.text_projection = nn.Sequential(
            nn.Linear(text_dim, 64 * 4 * 4),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.joint = nn.Sequential(
            nn.utils.spectral_norm(nn.Conv2d(512 + 64, 256, kernel_size=1)),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.head = nn.utils.spectral_norm(nn.Conv2d(256, 1, kernel_size=4))

    def forward(self, image, text_encoding, return_features=False):
        image_features = self.image_features(image)
        text_map = self.text_projection(text_encoding).view(image.size(0), 64, 4, 4)
        joint_features = self.joint(torch.cat([image_features, text_map], dim=1))
        logits = self.head(joint_features).view(-1)
        if return_features:
            return logits, joint_features
        return logits


def count_parameters(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if (p.requires_grad or not trainable_only))


text_encoder = TextEncoder(len(VOCAB)).to(DEVICE)
source_encoder = SourceEncoder().to(DEVICE)
generator = TargetGenerator().to(DEVICE)
discriminator = Discriminator().to(DEVICE)

encoder_params = count_parameters(source_encoder)
generator_params = count_parameters(generator)
print("Source encoder parameters:", encoder_params)
print("Generator parameters:", generator_params)
assert generator_params <= encoder_params / 2, "Generator must have at most half the parameters of SourceEncoder."


## Training

In [ ]:
def set_requires_grad(model, requires_grad):
    for parameter in model.parameters():
        parameter.requires_grad_(requires_grad)


def nt_xent_loss(z1, z2, temperature=0.2):
    batch_size = z1.size(0)
    if batch_size < 2:
        return z1.new_tensor(0.0)
    z = torch.cat([F.normalize(z1, dim=1), F.normalize(z2, dim=1)], dim=0)
    logits = z @ z.t() / temperature
    mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
    logits = logits.masked_fill(mask, -1e9)
    positives = torch.cat([
        torch.arange(batch_size, 2 * batch_size, device=z.device),
        torch.arange(0, batch_size, device=z.device),
    ])
    return F.cross_entropy(logits, positives)


def denormalize(image_tensor):
    return (image_tensor * 0.5 + 0.5).clamp(0, 1)


def train_gan(num_epochs=50, d_lr=2e-4, g_lr=2e-4, l1_weight=25.0, feature_weight=5.0, contrast_weight=0.1):
    d_optimizer = torch.optim.AdamW(discriminator.parameters(), lr=d_lr, betas=(0.5, 0.999), weight_decay=1e-4)
    g_optimizer = torch.optim.AdamW(
        list(source_encoder.parameters()) + list(text_encoder.parameters()) + list(generator.parameters()),
        lr=g_lr,
        betas=(0.5, 0.999),
        weight_decay=1e-4,
    )
    history = []

    for epoch in range(num_epochs):
        source_encoder.train()
        text_encoder.train()
        generator.train()
        discriminator.train()
        running = defaultdict(float)
        seen = 0

        for batch in train_loader:
            real_images = batch["image"].to(DEVICE)
            view1 = batch["view1"].to(DEVICE)
            view2 = batch["view2"].to(DEVICE)
            tokens = batch["tokens"].to(DEVICE)
            batch_size = real_images.size(0)
            seen += batch_size

            with torch.no_grad():
                text_for_d = text_encoder(tokens)
                source_rep_for_d = source_encoder(real_images)
                fake_for_d = generator(source_rep_for_d, text_for_d)
                wrong_text_for_d = torch.roll(text_for_d, shifts=1, dims=0)

            d_optimizer.zero_grad()
            real_logits = discriminator(real_images, text_for_d)
            fake_logits = discriminator(fake_for_d.detach(), text_for_d)
            wrong_logits = discriminator(real_images, wrong_text_for_d)
            d_loss = (
                F.relu(1.0 - real_logits).mean()
                + 0.5 * F.relu(1.0 + fake_logits).mean()
                + 0.5 * F.relu(1.0 + wrong_logits).mean()
            )
            d_loss.backward()
            d_optimizer.step()

            set_requires_grad(discriminator, False)
            g_optimizer.zero_grad()
            text_features = text_encoder(tokens)
            source_representation = source_encoder(real_images)
            fake_images = generator(source_representation, text_features)
            fake_logits, fake_features = discriminator(fake_images, text_features, return_features=True)
            _, real_features = discriminator(real_images, text_features.detach(), return_features=True)
            z1 = source_encoder(view1, project=True)
            z2 = source_encoder(view2, project=True)
            adv_loss = -fake_logits.mean()
            recon_loss = F.l1_loss(fake_images, real_images)
            fm_loss = F.l1_loss(fake_features, real_features.detach())
            contrast_loss = nt_xent_loss(z1, z2)
            g_loss = adv_loss + l1_weight * recon_loss + feature_weight * fm_loss + contrast_weight * contrast_loss
            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(source_encoder.parameters()) + list(text_encoder.parameters()) + list(generator.parameters()),
                max_norm=5.0,
            )
            g_optimizer.step()
            set_requires_grad(discriminator, True)

            running["d_loss"] += d_loss.item() * batch_size
            running["g_loss"] += g_loss.item() * batch_size
            running["recon_loss"] += recon_loss.item() * batch_size
            running["contrast_loss"] += contrast_loss.item() * batch_size

        epoch_log = {key: value / seen for key, value in running.items()}
        epoch_log["epoch"] = epoch + 1
        history.append(epoch_log)
        print(
            f"Epoch {epoch + 1:03d}/{num_epochs} | "
            f"D: {epoch_log['d_loss']:.4f} | G: {epoch_log['g_loss']:.4f} | "
            f"L1: {epoch_log['recon_loss']:.4f} | Contrast: {epoch_log['contrast_loss']:.4f}"
        )

        if (epoch + 1) % 10 == 0 or epoch == num_epochs - 1:
            torch.save(source_encoder.state_dict(), CHECKPOINT_DIR / "source_encoder.pth")
            torch.save(text_encoder.state_dict(), CHECKPOINT_DIR / "text_encoder.pth")
            torch.save(generator.state_dict(), CHECKPOINT_DIR / "target_generator.pth")
            torch.save(discriminator.state_dict(), CHECKPOINT_DIR / "discriminator.pth")

    pd.DataFrame(history).to_csv(CHECKPOINT_DIR / "training_history.csv", index=False)
    return history


NUM_EPOCHS = 50
assert NUM_EPOCHS <= 200
history = train_gan(num_epochs=NUM_EPOCHS)


In [ ]:
if history:
    history_df = pd.DataFrame(history)
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history_df["epoch"], history_df["d_loss"], label="D")
    plt.plot(history_df["epoch"], history_df["g_loss"], label="G")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(history_df["epoch"], history_df["recon_loss"], label="L1 reconstruction")
    plt.plot(history_df["epoch"], history_df["contrast_loss"], label="Contrastive")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(CHECKPOINT_DIR / "training_curves.png")
    plt.show()


## Testing

In [ ]:
def load_checkpoint_if_available(model, path):
    path = Path(path)
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        print("Loaded", path)


load_checkpoint_if_available(source_encoder, CHECKPOINT_DIR / "source_encoder.pth")
load_checkpoint_if_available(text_encoder, CHECKPOINT_DIR / "text_encoder.pth")
load_checkpoint_if_available(generator, CHECKPOINT_DIR / "target_generator.pth")
load_checkpoint_if_available(discriminator, CHECKPOINT_DIR / "discriminator.pth")

source_encoder.eval()
text_encoder.eval()
generator.eval()
discriminator.eval()


In [ ]:
def plot_test_generation_grid(samples_per_class=5):
    rows = []
    with torch.no_grad():
        for class_name in TEST_CLASSES:
            class_indices = [idx for idx, sample in enumerate(test_dataset.samples) if sample["class"] == class_name]
            chosen = random.Random(SEED + CLASS_TO_IDX[class_name]).sample(class_indices, min(samples_per_class, len(class_indices)))
            generated_for_class = []
            for idx in chosen:
                sample = test_dataset[idx]
                image = sample["image"].unsqueeze(0).to(DEVICE)
                tokens = sample["tokens"].unsqueeze(0).to(DEVICE)
                text_feature = text_encoder(tokens)
                source_representation = source_encoder(image)
                fake_image = generator(source_representation, text_feature)[0].cpu()
                generated_for_class.append((fake_image, sample["text"]))
            rows.append((class_name, generated_for_class))

    fig, axes = plt.subplots(len(rows), samples_per_class, figsize=(samples_per_class * 2.2, len(rows) * 2.2))
    if len(rows) == 1:
        axes = np.expand_dims(axes, axis=0)
    for row_idx, (class_name, generated_images) in enumerate(rows):
        for col_idx in range(samples_per_class):
            ax = axes[row_idx, col_idx]
            ax.axis("off")
            if col_idx < len(generated_images):
                image, text = generated_images[col_idx]
                ax.imshow(denormalize(image).permute(1, 2, 0).numpy())
                ax.set_title(class_name if col_idx == 0 else "", fontsize=8)
    plt.tight_layout()
    plt.savefig(CHECKPOINT_DIR / "test_generation_grid_5x5.png", dpi=160)
    plt.show()


plot_test_generation_grid(samples_per_class=5)


In [ ]:
def collect_encoder_embeddings(loader, max_points=None):
    embeddings, labels = [], []
    source_encoder.eval()
    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            reps = source_encoder(images).cpu().numpy()
            embeddings.append(reps)
            labels.extend(batch["class_id"].numpy().tolist())
            if max_points is not None and len(labels) >= max_points:
                break
    embeddings = np.concatenate(embeddings, axis=0)
    labels = np.asarray(labels)
    if max_points is not None:
        embeddings = embeddings[:max_points]
        labels = labels[:max_points]
    return embeddings, labels


TSNE_MAX_POINTS = None
embeddings, labels = collect_encoder_embeddings(all_selected_loader, max_points=TSNE_MAX_POINTS)
perplexity = min(30, max(5, (len(embeddings) - 1) // 3))
tsne_3d = TSNE(n_components=3, perplexity=perplexity, init="pca", learning_rate="auto", random_state=SEED).fit_transform(embeddings)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(tsne_3d[:, 0], tsne_3d[:, 1], tsne_3d[:, 2], c=labels, cmap="tab20", s=8, alpha=0.85)
ax.set_title("3D t-SNE of Source Encoder Embeddings")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_zlabel("t-SNE 3")
plt.colorbar(scatter, ax=ax, shrink=0.6, label="Selected class id")
plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / "source_encoder_tsne_3d.png", dpi=160)
plt.show()


In [ ]:
def model_size_mb_on_disk(model, filename):
    path = CHECKPOINT_DIR / filename
    torch.save(model.state_dict(), path)
    return path.stat().st_size / (1024 ** 2)


stats = []
for name, model, filename in [
    ("Source Encoder", source_encoder, "source_encoder_stats.pth"),
    ("Target Generator", generator, "target_generator_stats.pth"),
    ("Discriminator", discriminator, "discriminator_stats.pth"),
]:
    stats.append(
        {
            "Model": name,
            "Total Parameters": count_parameters(model),
            "Trainable Parameters": count_parameters(model, trainable_only=True),
            "Model Size on Disk (MB)": round(model_size_mb_on_disk(model, filename), 2),
        }
    )

stats_df = pd.DataFrame(stats)
display(stats_df)
stats_df.to_csv(CHECKPOINT_DIR / "model_stats.csv", index=False)


## Innovation Notes

- The source encoder is trained without class labels by adding an NT-Xent contrastive objective over two augmented views of the same image.
- The generator is deliberately parameter-efficient and checked against the half-source-encoder constraint before training.
- The discriminator uses spectral normalization and text-image mismatch negatives to improve conditional alignment.
- Feature matching and L1 reconstruction stabilize the GAN and make training more reliable within Colab resource limits.
